# Data Visualization for Bioinformatics

**A comprehensive guide to creating publication-quality biological figures**

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Matplotlib fundamentals** -- figures, axes, and all major plot types
2. **Customization** -- colors, labels, legends, annotations, styles
3. **Seaborn** -- statistical plots, heatmaps, pair plots, violin/box plots
4. **Plotly** -- interactive plots for exploratory analysis
5. **Bioinformatics visualizations** -- volcano plots, GC landscapes, expression heatmaps, PCA, and more
6. **Publication-quality figures** -- fonts, DPI, colorblind-friendly palettes, multi-panel layouts

---

| Section | Topic |
|---------|-------|
| 1 | Matplotlib Fundamentals |
| 2 | Plot Customization |
| 3 | Seaborn Statistical Plots |
| 4 | Bio: GC Content Sliding Window |
| 5 | Bio: Volcano Plot |
| 6 | Bio: Gene Expression Heatmap |
| 7 | Bio: Sequence Length Distribution |
| 8 | Bio: Quality Score Distribution |
| 9 | Bio: Genome Coverage Plot |
| 10 | Bio: PCA of Samples |
| 11 | Bio: Phylogenetic Dendrogram |
| 12 | Plotly Interactive Plots |
| 13 | Publication-Quality Figures |
| 14 | Exercises |

---

[← Previous: Module 17: Data Wrangling with Pandas](../../Tier_1_Python_for_Bioinformatics/17_Data_Wrangling/01_data_wrangling.ipynb) | [Next: Tier 2: Core Bioinformatics -- Skills Check →](../../Tier_2_Core_Bioinformatics/00_Skills_Check/00_skills_check.ipynb)

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy import stats
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.spatial.distance import pdist
from sklearn.decomposition import PCA

# Global style for the notebook
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('colorblind')
np.random.seed(42)

print('All libraries loaded successfully.')

---
## 1. Matplotlib Fundamentals

Matplotlib has two interfaces:
- **Pyplot (state-based)**: `plt.plot(...)` -- quick and simple
- **Object-oriented**: `fig, ax = plt.subplots()` then `ax.plot(...)` -- more control

For anything beyond a single quick plot, **always use the object-oriented interface**.

### 1.1 The Figure and Axes Model

```
Figure (the canvas)
 +-- Axes (a single plot area)
      +-- Title, xlabel, ylabel
      +-- x-axis, y-axis (ticks, limits)
      +-- plotted data (lines, bars, points)
```

In [ ]:
# Create a figure with one axes
fig, ax = plt.subplots(figsize=(8, 4))

# Simulated growth curve of bacterial culture (OD600 over time)
time_hours = np.linspace(0, 24, 200)
od600 = 0.05 * np.exp(0.3 * time_hours) / (1 + 0.05 * (np.exp(0.3 * time_hours) - 1) / 2.0)

ax.plot(time_hours, od600, color='darkgreen', linewidth=2)
ax.set_xlabel('Time (hours)', fontsize=12)
ax.set_ylabel('OD$_{600}$', fontsize=12)
ax.set_title('Bacterial Growth Curve (Logistic Model)', fontsize=14)
ax.set_xlim(0, 24)
ax.set_ylim(0, 2.2)

# Annotate growth phases
ax.axvspan(0, 3, alpha=0.15, color='gray', label='Lag')
ax.axvspan(3, 14, alpha=0.15, color='green', label='Exponential')
ax.axvspan(14, 24, alpha=0.15, color='orange', label='Stationary')
ax.legend(loc='upper left', fontsize=10)

plt.tight_layout()
plt.show()

### 1.2 Core Plot Types

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))

# --- Line plot: enzyme kinetics (Michaelis-Menten) ---
substrate = np.linspace(0, 100, 200)
vmax, km = 120, 15
velocity = vmax * substrate / (km + substrate)
axes[0, 0].plot(substrate, velocity, 'b-', linewidth=2)
axes[0, 0].axhline(vmax, color='red', linestyle='--', alpha=0.6, label=f'V_max = {vmax}')
axes[0, 0].axvline(km, color='gray', linestyle=':', alpha=0.6, label=f'K_m = {km}')
axes[0, 0].set_xlabel('[S] (mM)')
axes[0, 0].set_ylabel('Velocity (nmol/min)')
axes[0, 0].set_title('Line: Michaelis-Menten')
axes[0, 0].legend(fontsize=8)

# --- Scatter plot: gene expression correlation ---
gene_a = np.random.lognormal(3, 1.5, 200)
gene_b = gene_a * 0.7 + np.random.lognormal(2, 1, 200)
axes[0, 1].scatter(np.log2(gene_a + 1), np.log2(gene_b + 1),
                    alpha=0.5, s=15, c='steelblue', edgecolors='none')
axes[0, 1].set_xlabel('Gene A (log2 FPKM)')
axes[0, 1].set_ylabel('Gene B (log2 FPKM)')
axes[0, 1].set_title('Scatter: Expression Correlation')

# --- Bar plot: nucleotide composition ---
nucleotides = ['A', 'T', 'G', 'C']
human_freq = [29.3, 29.3, 20.7, 20.7]
ecoli_freq = [24.7, 23.6, 25.7, 26.0]
x_pos = np.arange(len(nucleotides))
width = 0.35
axes[0, 2].bar(x_pos - width/2, human_freq, width, label='Human', color='#2196F3')
axes[0, 2].bar(x_pos + width/2, ecoli_freq, width, label='E. coli', color='#FF9800')
axes[0, 2].set_xticks(x_pos)
axes[0, 2].set_xticklabels(nucleotides)
axes[0, 2].set_ylabel('Frequency (%)')
axes[0, 2].set_title('Bar: Nucleotide Composition')
axes[0, 2].legend(fontsize=8)

# --- Histogram: protein lengths ---
protein_lengths = np.concatenate([
    np.random.lognormal(5.5, 0.8, 800),
    np.random.lognormal(7, 0.5, 200)
])
axes[1, 0].hist(protein_lengths, bins=60, color='mediumpurple',
                edgecolor='white', linewidth=0.5, range=(0, 3000))
axes[1, 0].set_xlabel('Protein Length (aa)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title('Histogram: Protein Lengths')

# --- Pie chart: genome composition ---
labels = ['Exons', 'Introns', 'Intergenic', 'Repeats']
sizes = [1.5, 25, 28.5, 45]
colors = ['#4CAF50', '#2196F3', '#FFC107', '#F44336']
axes[1, 1].pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
               startangle=90, textprops={'fontsize': 9})
axes[1, 1].set_title('Pie: Human Genome Composition')

# --- Stem plot: SNP positions ---
snp_positions = np.sort(np.random.choice(range(1, 1001), 20, replace=False))
snp_effects = np.random.choice([-1, 0, 1], 20, p=[0.2, 0.6, 0.2])
colors_snp = ['red' if e == -1 else ('green' if e == 1 else 'gray') for e in snp_effects]
markerline, stemlines, baseline = axes[1, 2].stem(snp_positions, snp_effects)
plt.setp(stemlines, linewidth=1)
plt.setp(markerline, markersize=4)
axes[1, 2].set_xlabel('Genomic Position')
axes[1, 2].set_ylabel('Effect')
axes[1, 2].set_title('Stem: SNP Effects')
axes[1, 2].set_yticks([-1, 0, 1])
axes[1, 2].set_yticklabels(['Deleterious', 'Neutral', 'Beneficial'])

plt.tight_layout()
plt.show()

### 1.3 Subplots and Layout

Multiple approaches for arranging subplots:
- `plt.subplots(nrows, ncols)` -- regular grid
- `fig.add_gridspec()` -- flexible grid with spanning

In [ ]:
# GridSpec for flexible layout: one wide plot on top, two below
fig = plt.figure(figsize=(12, 7))
gs = fig.add_gridspec(2, 2, height_ratios=[1, 1], hspace=0.35, wspace=0.3)

ax_top = fig.add_subplot(gs[0, :])
ax_bl = fig.add_subplot(gs[1, 0])
ax_br = fig.add_subplot(gs[1, 1])

# Top: simulated expression timecourse for 3 genes
t = np.linspace(0, 48, 100)
for i, (name, phase) in enumerate([('p53', 0), ('MDM2', 4), ('CDKN1A', 8)]):
    signal = 2 * np.sin(2 * np.pi * t / 24 - phase) * np.exp(-t / 80) + 5
    ax_top.plot(t, signal, linewidth=2, label=name)
ax_top.set_xlabel('Time (hours)')
ax_top.set_ylabel('Expression (log2)')
ax_top.set_title('Gene Expression Timecourse After DNA Damage')
ax_top.legend()

# Bottom-left: bar chart of pathway enrichment
pathways = ['Apoptosis', 'Cell Cycle', 'DNA Repair', 'p53 Signaling', 'Autophagy']
neg_log_p = [8.5, 7.2, 6.1, 10.3, 3.2]
ax_bl.barh(pathways, neg_log_p, color=plt.cm.viridis(np.linspace(0.2, 0.8, 5)))
ax_bl.set_xlabel('-log$_{10}$(p-value)')
ax_bl.set_title('Pathway Enrichment')
ax_bl.axvline(x=-np.log10(0.05), color='red', linestyle='--', label='p=0.05')
ax_bl.legend(fontsize=8)

# Bottom-right: scatter of fold-change vs base mean
base_mean = np.random.lognormal(5, 2, 500)
log2fc = np.random.normal(0, 0.5, 500)
log2fc[:30] = np.random.normal(2, 0.3, 30)
ax_br.scatter(np.log10(base_mean), log2fc, s=10, alpha=0.5, c='gray')
ax_br.scatter(np.log10(base_mean[:30]), log2fc[:30], s=15, alpha=0.8, c='red', label='Significant')
ax_br.set_xlabel('log$_{10}$(Base Mean)')
ax_br.set_ylabel('log$_2$(Fold Change)')
ax_br.set_title('MA Plot')
ax_br.axhline(0, color='black', linewidth=0.8)
ax_br.legend(fontsize=8)

plt.show()

---
## 2. Plot Customization

Making plots informative and visually appealing.

In [ ]:
# Demonstrate customization on a single figure
fig, ax = plt.subplots(figsize=(9, 5))

# Simulated RT-qPCR data: expression of 5 genes in control vs treatment
genes = ['GAPDH', 'TP53', 'BRCA1', 'MYC', 'BCL2']
control = [1.0, 1.0, 1.0, 1.0, 1.0]
treatment = [1.05, 3.2, 0.4, 5.1, 0.25]
treatment_err = [0.1, 0.5, 0.08, 0.8, 0.05]

x = np.arange(len(genes))
width = 0.35

bars1 = ax.bar(x - width/2, control, width, label='Control',
               color='#90CAF9', edgecolor='#1565C0', linewidth=1.2)
bars2 = ax.bar(x + width/2, treatment, width, yerr=treatment_err,
               label='Treatment (6h cisplatin)',
               color='#EF9A9A', edgecolor='#C62828', linewidth=1.2,
               capsize=4)

# Customization
ax.set_xticks(x)
ax.set_xticklabels(genes, fontsize=12, fontweight='bold')
ax.set_ylabel('Relative Expression\n(fold change)', fontsize=12)
ax.set_title('RT-qPCR: Gene Expression After Cisplatin Treatment', fontsize=14, pad=15)
ax.legend(fontsize=11, frameon=True, fancybox=True, shadow=True)
ax.axhline(1, color='gray', linestyle='--', linewidth=0.8, zorder=0)

# Add significance stars
for i, (c, t) in enumerate(zip(control, treatment)):
    if abs(t - c) > 0.5:
        y_pos = max(c, t + treatment_err[i]) + 0.3
        ax.text(i + width/2, y_pos, '*', ha='center', fontsize=16, color='red')

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.set_ylim(0, 7)

plt.tight_layout()
plt.show()

### 2.1 Colors, Colormaps, and Colorblind-Friendly Palettes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# 1. Named color palettes for categorical data
palettes = {
    'colorblind': sns.color_palette('colorblind', 6),
    'Set2': sns.color_palette('Set2', 6),
    'deep': sns.color_palette('deep', 6),
}
for row, (name, pal) in enumerate(palettes.items()):
    for i, c in enumerate(pal):
        axes[0].add_patch(plt.Rectangle((i, row), 0.9, 0.9, color=c))
axes[0].set_xlim(-0.2, 6.2)
axes[0].set_ylim(-0.2, 3.2)
axes[0].set_yticks([0.45, 1.45, 2.45])
axes[0].set_yticklabels(list(palettes.keys()))
axes[0].set_title('Categorical Palettes')
axes[0].set_xticks([])

# 2. Sequential colormap (great for heatmaps)
gradient = np.linspace(0, 1, 256).reshape(1, -1)
for row, cmap_name in enumerate(['viridis', 'plasma', 'cividis']):
    axes[1].imshow(gradient, aspect='auto', cmap=cmap_name,
                   extent=[0, 10, row, row+0.8])
axes[1].set_yticks([0.4, 1.4, 2.4])
axes[1].set_yticklabels(['viridis', 'plasma', 'cividis'])
axes[1].set_title('Sequential Colormaps (perceptually uniform)')
axes[1].set_xticks([])
axes[1].set_ylim(-0.2, 3)

# 3. Diverging colormap (centered data like fold-change)
diverging = np.linspace(-1, 1, 256).reshape(1, -1)
for row, cmap_name in enumerate(['RdBu_r', 'coolwarm', 'PiYG']):
    axes[2].imshow(diverging, aspect='auto', cmap=cmap_name,
                   extent=[0, 10, row, row+0.8])
axes[2].set_yticks([0.4, 1.4, 2.4])
axes[2].set_yticklabels(['RdBu_r', 'coolwarm', 'PiYG'])
axes[2].set_title('Diverging Colormaps (centered at 0)')
axes[2].set_xticks([])
axes[2].set_ylim(-0.2, 3)

plt.tight_layout()
plt.show()

print('Tip: viridis, cividis, and the colorblind palette are safe for color vision deficiency.')

---
## 3. Seaborn Statistical Plots

Seaborn is built on Matplotlib and provides a higher-level interface for statistical graphics. It integrates closely with Pandas DataFrames.

In [ ]:
# Create a realistic gene expression DataFrame
np.random.seed(42)
n_samples = 200

cell_types = np.random.choice(['B cell', 'T cell', 'Macrophage', 'Neuron'], n_samples)
conditions = np.random.choice(['Control', 'Stimulated'], n_samples)

expr_df = pd.DataFrame({
    'Cell_Type': cell_types,
    'Condition': conditions,
    'CD19': np.where(cell_types == 'B cell',
                     np.random.normal(8, 1, n_samples),
                     np.random.normal(1, 0.5, n_samples)),
    'CD3': np.where(cell_types == 'T cell',
                    np.random.normal(9, 0.8, n_samples),
                    np.random.normal(0.5, 0.3, n_samples)),
    'TNF_alpha': np.where((cell_types == 'Macrophage') & (conditions == 'Stimulated'),
                          np.random.normal(10, 1.5, n_samples),
                          np.random.normal(3, 1, n_samples)),
    'NEFL': np.where(cell_types == 'Neuron',
                     np.random.normal(7, 1.2, n_samples),
                     np.random.normal(0.2, 0.3, n_samples)),
})

print(f'DataFrame shape: {expr_df.shape}')
expr_df.head()

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Box plot: expression by cell type
sns.boxplot(data=expr_df, x='Cell_Type', y='TNF_alpha', hue='Condition',
            ax=axes[0, 0], palette='Set2')
axes[0, 0].set_title('Box Plot: TNF-alpha Expression')
axes[0, 0].set_ylabel('Expression (log2 FPKM)')

# Violin plot: shows full distribution shape
sns.violinplot(data=expr_df, x='Cell_Type', y='CD19', hue='Condition',
               split=True, ax=axes[0, 1], palette='muted', inner='quartile')
axes[0, 1].set_title('Violin Plot: CD19 Expression')
axes[0, 1].set_ylabel('Expression (log2 FPKM)')

# Strip plot: individual data points
sns.stripplot(data=expr_df, x='Cell_Type', y='CD3', hue='Condition',
              dodge=True, ax=axes[1, 0], palette='deep', alpha=0.6, size=4)
axes[1, 0].set_title('Strip Plot: CD3 Expression')
axes[1, 0].set_ylabel('Expression (log2 FPKM)')

# KDE plot: density estimation
for ct in ['B cell', 'Neuron', 'Macrophage']:
    subset = expr_df[expr_df['Cell_Type'] == ct]
    sns.kdeplot(data=subset, x='NEFL', ax=axes[1, 1], label=ct, fill=True, alpha=0.3)
axes[1, 1].set_title('KDE Plot: NEFL Expression by Cell Type')
axes[1, 1].set_xlabel('Expression (log2 FPKM)')
axes[1, 1].legend()

plt.tight_layout()
plt.show()

### 3.1 Pair Plot: Multivariate Relationships at a Glance

In [ ]:
# Pair plot colored by cell type (subset for speed)
g = sns.pairplot(expr_df[['Cell_Type', 'CD19', 'CD3', 'TNF_alpha', 'NEFL']],
                 hue='Cell_Type', palette='colorblind',
                 plot_kws={'alpha': 0.5, 's': 15},
                 diag_kind='kde', height=2.2)
g.fig.suptitle('Pair Plot: Marker Gene Expression by Cell Type', y=1.02, fontsize=14)
plt.show()

### 3.2 Joint Plot: Bivariate Distribution

In [ ]:
g = sns.jointplot(data=expr_df, x='CD19', y='CD3',
                  hue='Cell_Type', palette='colorblind',
                  alpha=0.5, height=7)
g.fig.suptitle('Joint Distribution: CD19 vs CD3', y=1.02)
plt.show()

---
## 4. Bio: GC Content Sliding Window Plot

GC content varies along chromosomes. GC-rich regions often correspond to gene-dense areas, CpG islands, and specific chromatin states.

In [ ]:
def generate_biased_sequence(length, gc_content):
    """Generate a DNA sequence with specified approximate GC content."""
    gc = gc_content / 100
    at = 1 - gc
    weights = [at/2, at/2, gc/2, gc/2]  # A, T, G, C
    return ''.join(np.random.choice(['A', 'T', 'G', 'C'], length, p=weights))

def sliding_gc(sequence, window_size=100, step=10):
    """Calculate GC% in a sliding window along a sequence."""
    positions, gc_values = [], []
    for i in range(0, len(sequence) - window_size + 1, step):
        window = sequence[i:i + window_size]
        gc = (window.count('G') + window.count('C')) / window_size * 100
        positions.append(i + window_size // 2)
        gc_values.append(gc)
    return np.array(positions), np.array(gc_values)

# Build a sequence with distinct GC regions (like a real chromosome)
np.random.seed(42)
regions = [
    (800, 38),   # AT-rich region (heterochromatin-like)
    (200, 72),   # CpG island
    (600, 45),   # moderate
    (150, 75),   # CpG island
    (500, 40),   # AT-rich
    (250, 68),   # GC-rich gene cluster
    (500, 42),   # AT-rich
]
sequence = ''.join(generate_biased_sequence(l, gc) for l, gc in regions)
positions, gc_values = sliding_gc(sequence, window_size=100, step=10)

print(f'Sequence length: {len(sequence):,} bp')
print(f'Overall GC: {(sequence.count("G") + sequence.count("C")) / len(sequence) * 100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

# Fill regions above and below 50%
ax.fill_between(positions, gc_values, 50,
                where=(gc_values >= 50), color='#E53935', alpha=0.4, label='GC-rich (>50%)')
ax.fill_between(positions, gc_values, 50,
                where=(gc_values < 50), color='#1E88E5', alpha=0.4, label='AT-rich (<50%)')
ax.plot(positions, gc_values, 'k-', linewidth=0.8)
ax.axhline(50, color='gray', linestyle='--', linewidth=1)

# Mark CpG island regions
cpg_starts = [800, 1600]  # approximate CpG island starts
cpg_widths = [200, 150]
for s, w in zip(cpg_starts, cpg_widths):
    ax.annotate('CpG island', xy=(s + w/2, 78), fontsize=8,
                ha='center', color='darkred', fontweight='bold')

ax.set_xlabel('Position (bp)', fontsize=12)
ax.set_ylabel('GC Content (%)', fontsize=12)
ax.set_title('GC Content Sliding Window (100 bp window, 10 bp step)', fontsize=13)
ax.legend(loc='lower right', fontsize=10)
ax.set_xlim(0, len(sequence))
ax.set_ylim(15, 90)

plt.tight_layout()
plt.show()

---
## 5. Bio: Volcano Plot

The volcano plot is the standard visualization for differential gene expression: **log2 fold-change** on x-axis vs. **-log10(p-value)** on y-axis. Genes that are both statistically significant and biologically meaningful appear in the upper corners.

In [ ]:
np.random.seed(42)
n_genes = 5000

# Background genes: small effect, high p-value
log2fc = np.random.normal(0, 0.4, n_genes)
pvalues = np.random.uniform(0.01, 1, n_genes)

# Inject upregulated genes
n_up = 80
up_idx = np.random.choice(n_genes, n_up, replace=False)
log2fc[up_idx] = np.abs(np.random.normal(2.5, 0.8, n_up))
pvalues[up_idx] = 10 ** np.random.uniform(-12, -2, n_up)

# Inject downregulated genes
n_down = 60
remaining = list(set(range(n_genes)) - set(up_idx))
down_idx = np.random.choice(remaining, n_down, replace=False)
log2fc[down_idx] = -np.abs(np.random.normal(2, 0.7, n_down))
pvalues[down_idx] = 10 ** np.random.uniform(-10, -2, n_down)

# Build DataFrame
gene_names = [f'Gene_{i}' for i in range(n_genes)]
# Give some real-sounding names to top hits
real_names = ['TP53', 'BRCA1', 'MYC', 'BCL2', 'EGFR', 'VEGFA', 'KRAS', 'PTEN', 'RB1', 'APC']
top_indices = np.argsort(pvalues)[:len(real_names)]
for i, idx in enumerate(top_indices):
    gene_names[idx] = real_names[i]

volcano_df = pd.DataFrame({
    'gene': gene_names,
    'log2FC': log2fc,
    'pvalue': pvalues,
    'neg_log10_p': -np.log10(pvalues)
})

# Classify
fc_thresh, p_thresh = 1.0, 0.05
volcano_df['category'] = 'Not Significant'
volcano_df.loc[(volcano_df['log2FC'] > fc_thresh) & (volcano_df['pvalue'] < p_thresh), 'category'] = 'Up'
volcano_df.loc[(volcano_df['log2FC'] < -fc_thresh) & (volcano_df['pvalue'] < p_thresh), 'category'] = 'Down'

counts = volcano_df['category'].value_counts()
print(f"Up: {counts.get('Up', 0)}, Down: {counts.get('Down', 0)}, NS: {counts.get('Not Significant', 0)}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))

color_map = {'Not Significant': '#BDBDBD', 'Up': '#E53935', 'Down': '#1E88E5'}

for cat in ['Not Significant', 'Up', 'Down']:
    sub = volcano_df[volcano_df['category'] == cat]
    ax.scatter(sub['log2FC'], sub['neg_log10_p'],
              c=color_map[cat], s=12, alpha=0.6, label=f'{cat} ({len(sub)})', edgecolors='none')

# Threshold lines
ax.axhline(-np.log10(p_thresh), color='black', linestyle='--', linewidth=0.8)
ax.axvline(fc_thresh, color='black', linestyle='--', linewidth=0.8)
ax.axvline(-fc_thresh, color='black', linestyle='--', linewidth=0.8)

# Label top hits
to_label = volcano_df.nlargest(8, 'neg_log10_p')
for _, row in to_label.iterrows():
    ax.annotate(row['gene'],
                xy=(row['log2FC'], row['neg_log10_p']),
                xytext=(5, 5), textcoords='offset points',
                fontsize=8, fontweight='bold',
                arrowprops=dict(arrowstyle='-', color='gray', lw=0.5))

ax.set_xlabel('log$_2$(Fold Change)', fontsize=13)
ax.set_ylabel('-log$_{10}$(p-value)', fontsize=13)
ax.set_title('Volcano Plot: Differential Gene Expression\n(Treatment vs Control)', fontsize=14)
ax.legend(title='Category', fontsize=10, title_fontsize=11)

plt.tight_layout()
plt.show()

---
## 6. Bio: Gene Expression Heatmap with Clustering

Heatmaps with hierarchical clustering are essential for visualizing expression patterns across genes and samples.

In [ ]:
np.random.seed(42)

n_genes_hm = 40
samples = (['Ctrl_1', 'Ctrl_2', 'Ctrl_3', 'Ctrl_4'] +
           ['Drug_1', 'Drug_2', 'Drug_3', 'Drug_4'])

# Base expression (log-normal)
expression = np.random.lognormal(mean=5, sigma=1, size=(n_genes_hm, 8))

# Create gene clusters with distinct behavior
# Cluster 1 (genes 0-9): upregulated in drug
expression[:10, 4:] *= np.random.uniform(3, 6, (10, 4))
# Cluster 2 (genes 10-19): downregulated in drug
expression[10:20, 4:] *= np.random.uniform(0.1, 0.3, (10, 4))
# Cluster 3 (genes 20-30): unchanged
# Cluster 4 (genes 30-40): mixed response
expression[30:35, 4:] *= np.random.uniform(2, 4, (5, 4))
expression[35:40, 4:] *= np.random.uniform(0.2, 0.5, (5, 4))

# Z-score normalize across samples (row-wise)
expr_z = (expression - expression.mean(axis=1, keepdims=True)) / expression.std(axis=1, keepdims=True)

gene_labels = ([f'UP_{i}' for i in range(10)] +
               [f'DOWN_{i}' for i in range(10)] +
               [f'STABLE_{i}' for i in range(10)] +
               [f'MIXED_{i}' for i in range(10)])

expr_matrix = pd.DataFrame(expr_z, index=gene_labels, columns=samples)
print(f'Expression matrix: {expr_matrix.shape[0]} genes x {expr_matrix.shape[1]} samples')

In [ ]:
# Clustered heatmap with condition color bar
condition_colors = ['#1E88E5'] * 4 + ['#E53935'] * 4  # blue=ctrl, red=drug
col_colors = pd.Series(condition_colors, index=samples, name='Condition')

g = sns.clustermap(expr_matrix,
                   cmap='RdBu_r', center=0, vmin=-2.5, vmax=2.5,
                   figsize=(10, 12),
                   col_colors=col_colors,
                   dendrogram_ratio=(0.12, 0.08),
                   cbar_pos=(0.02, 0.82, 0.03, 0.12),
                   linewidths=0.3, linecolor='white',
                   yticklabels=True, xticklabels=True)

g.ax_heatmap.set_xlabel('Samples', fontsize=12)
g.ax_heatmap.set_ylabel('Genes', fontsize=12)
g.fig.suptitle('Gene Expression Heatmap (Z-score normalized)', fontsize=14, y=1.01)

# Add legend for condition colors
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='#1E88E5', label='Control'),
                   Patch(facecolor='#E53935', label='Drug')]
g.ax_heatmap.legend(handles=legend_elements, loc='lower right',
                    bbox_to_anchor=(1.3, 1.02), fontsize=9)

plt.show()

---
## 7. Bio: Sequence Length Distribution

Histograms of sequence lengths are fundamental in genomics -- for contigs, reads, transcripts, or proteins.

In [ ]:
np.random.seed(42)

# Simulate transcript lengths from different sources
mrna_lengths = np.random.lognormal(mean=7.0, sigma=0.9, size=3000).astype(int)
lncrna_lengths = np.random.lognormal(mean=6.5, sigma=1.1, size=1500).astype(int)
mirna_lengths = np.random.normal(loc=22, scale=2, size=500).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: overlapping histograms
axes[0].hist(mrna_lengths, bins=80, alpha=0.6, label=f'mRNA (n={len(mrna_lengths)})',
             color='#2196F3', range=(0, 8000), density=True)
axes[0].hist(lncrna_lengths, bins=80, alpha=0.6, label=f'lncRNA (n={len(lncrna_lengths)})',
             color='#FF9800', range=(0, 8000), density=True)
axes[0].set_xlabel('Transcript Length (nt)', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Transcript Length Distribution', fontsize=13)
axes[0].legend(fontsize=10)
axes[0].set_xlim(0, 8000)

# Add median lines
for data, color, label in [(mrna_lengths, '#2196F3', 'mRNA'),
                            (lncrna_lengths, '#FF9800', 'lncRNA')]:
    med = np.median(data)
    axes[0].axvline(med, color=color, linestyle='--', linewidth=1.5)
    axes[0].text(med + 100, axes[0].get_ylim()[1] * 0.9,
                 f'{label} median\n{med:.0f} nt', fontsize=8, color=color)

# Right: miRNA lengths (much shorter, different scale)
axes[1].hist(mirna_lengths, bins=range(15, 35), color='#4CAF50',
             edgecolor='white', linewidth=0.8, alpha=0.8)
axes[1].set_xlabel('miRNA Length (nt)', fontsize=12)
axes[1].set_ylabel('Count', fontsize=12)
axes[1].set_title(f'miRNA Length Distribution (n={len(mirna_lengths)})', fontsize=13)
axes[1].axvline(22, color='red', linestyle='--', linewidth=1.5, label='Canonical 22 nt')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.show()

---
## 8. Bio: Quality Score Distribution (Sequencing Data)

Per-base quality (Phred score) is critical for assessing sequencing run quality. This mimics a FastQC-style plot.

In [ ]:
np.random.seed(42)
read_length = 150
n_reads = 1000

# Simulate quality scores: high in the middle, dropping at the end
base_positions = np.arange(1, read_length + 1)

quality_data = []
for pos in base_positions:
    if pos <= 5:
        # First few bases: slightly lower quality
        q = np.random.normal(32, 3, n_reads)
    elif pos <= 100:
        # Good quality region
        q = np.random.normal(36, 2, n_reads)
    elif pos <= 130:
        # Gradual decline
        mean_q = 36 - (pos - 100) * 0.3
        q = np.random.normal(mean_q, 3, n_reads)
    else:
        # Tail quality drop
        mean_q = 27 - (pos - 130) * 0.5
        q = np.random.normal(mean_q, 5, n_reads)
    quality_data.append(np.clip(q, 2, 41).astype(int))

quality_array = np.array(quality_data)  # shape: (150, 1000)

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Box plot style: show median, Q1, Q3, and whiskers at every 5th position
step = 3
positions_plot = base_positions[::step]
box_data = [quality_array[i, :] for i in range(0, read_length, step)]

bp = ax.boxplot(box_data, positions=positions_plot, widths=2,
                patch_artist=True, showfliers=False,
                medianprops=dict(color='red', linewidth=1.5))

# Color boxes by median quality
for i, box in enumerate(bp['boxes']):
    median = np.median(box_data[i])
    if median >= 30:
        box.set_facecolor('#A5D6A7')  # green = good
    elif median >= 20:
        box.set_facecolor('#FFF176')  # yellow = acceptable
    else:
        box.set_facecolor('#EF9A9A')  # red = poor
    box.set_alpha(0.8)

# Quality thresholds
ax.axhline(30, color='green', linestyle='--', linewidth=1, alpha=0.7, label='Q30 (good)')
ax.axhline(20, color='orange', linestyle='--', linewidth=1, alpha=0.7, label='Q20 (acceptable)')

# Background shading
ax.axhspan(30, 42, alpha=0.05, color='green')
ax.axhspan(20, 30, alpha=0.05, color='yellow')
ax.axhspan(0, 20, alpha=0.05, color='red')

ax.set_xlabel('Position in Read (bp)', fontsize=12)
ax.set_ylabel('Phred Quality Score', fontsize=12)
ax.set_title('Per-Base Sequence Quality (FastQC-style)', fontsize=14)
ax.set_ylim(0, 42)
ax.set_xlim(0, read_length + 1)
ax.legend(loc='lower left', fontsize=10)

plt.tight_layout()
plt.show()

---
## 9. Bio: Genome Coverage Plot

Coverage plots show the depth of sequencing reads aligned to a reference. They help identify well-covered regions, gaps, and potential CNVs.

In [ ]:
np.random.seed(42)
region_length = 5000  # 5 kb region

# Base coverage ~50x with Poisson noise
base_coverage = np.random.poisson(50, region_length).astype(float)

# Add a deletion (coverage drop to ~5x)
base_coverage[1500:2000] = np.random.poisson(5, 500)

# Add a duplication (coverage ~100x)
base_coverage[3200:3700] = np.random.poisson(100, 500)

# Smooth for visualization
window = 20
smoothed = np.convolve(base_coverage, np.ones(window)/window, mode='same')

# Gene annotation
genes = [
    {'name': 'GENE_A', 'start': 200, 'end': 1200, 'strand': '+'},
    {'name': 'GENE_B', 'start': 1400, 'end': 2800, 'strand': '-'},
    {'name': 'GENE_C', 'start': 3000, 'end': 4500, 'strand': '+'},
]

In [ ]:
fig, (ax_cov, ax_gene) = plt.subplots(2, 1, figsize=(14, 6),
                                       height_ratios=[4, 1],
                                       sharex=True, gridspec_kw={'hspace': 0.05})

# Coverage plot
positions = np.arange(region_length)
ax_cov.fill_between(positions, smoothed, alpha=0.3, color='steelblue')
ax_cov.plot(positions, smoothed, color='steelblue', linewidth=0.8)

# Expected coverage line
ax_cov.axhline(50, color='black', linestyle='--', linewidth=0.8, alpha=0.5, label='Expected (50x)')

# Highlight CNV regions
ax_cov.axvspan(1500, 2000, alpha=0.15, color='red')
ax_cov.text(1750, 110, 'Deletion', ha='center', fontsize=10, color='red', fontweight='bold')
ax_cov.axvspan(3200, 3700, alpha=0.15, color='blue')
ax_cov.text(3450, 110, 'Duplication', ha='center', fontsize=10, color='blue', fontweight='bold')

ax_cov.set_ylabel('Coverage (x)', fontsize=12)
ax_cov.set_title('Genome Coverage Plot with Structural Variants', fontsize=14)
ax_cov.legend(loc='upper right', fontsize=10)
ax_cov.set_ylim(0, 130)

# Gene annotation track
gene_colors = ['#4CAF50', '#2196F3', '#FF9800']
for i, gene in enumerate(genes):
    y_center = 0.5
    arrow_dir = 1 if gene['strand'] == '+' else -1
    ax_gene.annotate('', xy=(gene['end'], y_center), xytext=(gene['start'], y_center),
                     arrowprops=dict(arrowstyle='->', color=gene_colors[i], lw=8,
                                    mutation_scale=15))
    mid = (gene['start'] + gene['end']) / 2
    ax_gene.text(mid, 0.1, f"{gene['name']} ({gene['strand']})",
                 ha='center', fontsize=9, fontweight='bold')

ax_gene.set_ylim(-0.2, 1.2)
ax_gene.set_xlim(0, region_length)
ax_gene.set_xlabel('Genomic Position (bp)', fontsize=12)
ax_gene.set_yticks([])
ax_gene.set_ylabel('Genes', fontsize=10)
ax_gene.spines['top'].set_visible(False)
ax_gene.spines['right'].set_visible(False)
ax_gene.spines['left'].set_visible(False)

plt.tight_layout()
plt.show()

---
## 10. Bio: PCA of Samples Colored by Condition

Principal Component Analysis (PCA) reduces high-dimensional expression data to 2D for visualizing sample relationships and batch effects.

In [ ]:
np.random.seed(42)

n_genes_pca = 500
sample_labels = (['WT_rep1', 'WT_rep2', 'WT_rep3', 'WT_rep4'] +
                 ['KO_rep1', 'KO_rep2', 'KO_rep3', 'KO_rep4'] +
                 ['Drug_rep1', 'Drug_rep2', 'Drug_rep3', 'Drug_rep4'])
conditions = ['Wild Type'] * 4 + ['Knockout'] * 4 + ['Drug Treated'] * 4

# Simulate expression: each condition has a shifted mean for some genes
base_expr = np.random.lognormal(5, 1.5, (n_genes_pca, 12))

# Condition effects on subsets of genes
base_expr[:100, 4:8] *= 3     # KO upregulates first 100 genes
base_expr[100:180, 4:8] *= 0.2  # KO downregulates genes 100-180
base_expr[200:320, 8:12] *= 4   # Drug upregulates genes 200-320
base_expr[50:150, 8:12] *= 0.3  # Drug downregulates genes 50-150

# Log transform
log_expr = np.log2(base_expr + 1)

# PCA
pca = PCA(n_components=2)
pcs = pca.fit_transform(log_expr.T)  # samples in rows

pca_df = pd.DataFrame({
    'PC1': pcs[:, 0], 'PC2': pcs[:, 1],
    'Sample': sample_labels, 'Condition': conditions
})

print(f'Variance explained: PC1={pca.explained_variance_ratio_[0]*100:.1f}%, '
      f'PC2={pca.explained_variance_ratio_[1]*100:.1f}%')

In [ ]:
fig, ax = plt.subplots(figsize=(9, 7))

palette = {'Wild Type': '#2196F3', 'Knockout': '#E53935', 'Drug Treated': '#4CAF50'}
markers = {'Wild Type': 'o', 'Knockout': 's', 'Drug Treated': '^'}

for cond in palette:
    sub = pca_df[pca_df['Condition'] == cond]
    ax.scatter(sub['PC1'], sub['PC2'], c=palette[cond], marker=markers[cond],
              s=120, label=cond, edgecolors='black', linewidth=0.8, zorder=3)
    # Draw ellipse (confidence region)
    from matplotlib.patches import Ellipse
    mean_x, mean_y = sub['PC1'].mean(), sub['PC2'].mean()
    std_x, std_y = sub['PC1'].std(), sub['PC2'].std()
    ellipse = Ellipse((mean_x, mean_y), width=std_x*3, height=std_y*3,
                      fill=True, alpha=0.1, color=palette[cond], zorder=1)
    ax.add_patch(ellipse)

# Sample labels
for _, row in pca_df.iterrows():
    ax.annotate(row['Sample'], (row['PC1'], row['PC2']),
                fontsize=7, ha='left', va='bottom',
                xytext=(4, 4), textcoords='offset points')

ax.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)", fontsize=12)
ax.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)", fontsize=12)
ax.set_title('PCA of RNA-seq Samples', fontsize=14)
ax.legend(fontsize=11, frameon=True)
ax.axhline(0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.5, alpha=0.5)

plt.tight_layout()
plt.show()

---
## 11. Bio: Phylogenetic-Style Dendrogram

Dendrograms visualize hierarchical relationships -- between species, samples, or gene clusters.

In [ ]:
# Simulated distance matrix for species (based on sequence divergence)
species = ['Human', 'Chimpanzee', 'Gorilla', 'Orangutan', 'Gibbon',
           'Macaque', 'Mouse', 'Rat', 'Dog', 'Cow', 'Chicken', 'Zebrafish']

# Create a realistic-ish distance matrix (smaller = more similar)
np.random.seed(42)
n_sp = len(species)

# Build distances reflecting known phylogeny
dist_matrix = np.zeros((n_sp, n_sp))
# Great apes are close
groups = {
    'great_apes': [0, 1, 2, 3],   # Human, Chimp, Gorilla, Orangutan
    'other_primates': [4, 5],       # Gibbon, Macaque
    'rodents': [6, 7],              # Mouse, Rat
    'other_mammals': [8, 9],        # Dog, Cow
    'non_mammals': [10, 11],        # Chicken, Zebrafish
}

for i in range(n_sp):
    for j in range(i+1, n_sp):
        # Base distance
        d = 50 + np.random.normal(0, 2)
        # Within great apes
        if i in groups['great_apes'] and j in groups['great_apes']:
            d = 5 + np.random.normal(0, 1)
            if {i, j} == {0, 1}:  # Human-Chimp closest
                d = 1.5
        # Within rodents
        elif i in groups['rodents'] and j in groups['rodents']:
            d = 8 + np.random.normal(0, 1)
        # Primates to other primates
        elif (i in groups['great_apes'] and j in groups['other_primates']) or \
             (j in groups['great_apes'] and i in groups['other_primates']):
            d = 15 + np.random.normal(0, 2)
        # Mammals to mammals
        elif i in groups['other_mammals'] and j in groups['other_mammals']:
            d = 20 + np.random.normal(0, 2)
        # Non-mammals
        elif i in groups['non_mammals'] and j in groups['non_mammals']:
            d = 35 + np.random.normal(0, 2)
        elif i in groups['non_mammals'] or j in groups['non_mammals']:
            d = 60 + np.random.normal(0, 3)
        dist_matrix[i, j] = dist_matrix[j, i] = abs(d)

# Convert to condensed form and compute linkage
condensed = pdist(dist_matrix)
Z = linkage(condensed, method='average')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Standard dendrogram
dn = dendrogram(Z, labels=species, ax=axes[0], leaf_rotation=45,
                leaf_font_size=10, color_threshold=25,
                above_threshold_color='gray')
axes[0].set_ylabel('Distance (sequence divergence)', fontsize=11)
axes[0].set_title('Phylogenetic Dendrogram (top-down)', fontsize=13)

# Horizontal dendrogram (tree-like)
dn2 = dendrogram(Z, labels=species, ax=axes[1], orientation='left',
                 leaf_font_size=10, color_threshold=25,
                 above_threshold_color='gray')
axes[1].set_xlabel('Distance (sequence divergence)', fontsize=11)
axes[1].set_title('Phylogenetic Dendrogram (horizontal)', fontsize=13)

plt.tight_layout()
plt.show()

---
## 12. Plotly: Interactive Plots

Plotly creates interactive, zoomable, hoverable plots -- ideal for exploratory analysis in Jupyter notebooks.

In [ ]:
try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
    print('Plotly loaded successfully.')
except ImportError:
    PLOTLY_AVAILABLE = False
    print('Plotly not installed. Install with: pip install plotly')
    print('Skipping interactive plot sections (static alternatives shown).')

In [ ]:
if PLOTLY_AVAILABLE:
    # Interactive volcano plot
    fig_plotly = px.scatter(
        volcano_df, x='log2FC', y='neg_log10_p',
        color='category',
        color_discrete_map={'Not Significant': '#BDBDBD', 'Up': '#E53935', 'Down': '#1E88E5'},
        hover_data=['gene', 'log2FC', 'pvalue'],
        title='Interactive Volcano Plot (hover over points!)',
        labels={'log2FC': 'log2(Fold Change)', 'neg_log10_p': '-log10(p-value)'},
        opacity=0.6
    )
    fig_plotly.add_hline(y=-np.log10(0.05), line_dash='dash', line_color='black')
    fig_plotly.add_vline(x=1, line_dash='dash', line_color='black')
    fig_plotly.add_vline(x=-1, line_dash='dash', line_color='black')
    fig_plotly.update_layout(width=800, height=600)
    fig_plotly.show()
else:
    print('Plotly not available -- see the static volcano plot in Section 5.')

In [ ]:
if PLOTLY_AVAILABLE:
    # Interactive PCA
    fig_pca = px.scatter(
        pca_df, x='PC1', y='PC2',
        color='Condition', text='Sample',
        color_discrete_map=palette,
        title='Interactive PCA (drag to zoom, hover for details)',
        labels={
            'PC1': f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)',
            'PC2': f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)'
        }
    )
    fig_pca.update_traces(textposition='top center', marker=dict(size=12))
    fig_pca.update_layout(width=700, height=550)
    fig_pca.show()
else:
    print('Plotly not available -- see the static PCA plot in Section 10.')

---
## 13. Publication-Quality Figures

Journals have strict requirements for figures: high DPI, specific fonts, vector formats, etc.

### 13.1 Setting Publication Defaults

In [ ]:
# Publication-quality RC parameters
pub_params = {
    'figure.figsize': (7, 5),       # Nature single column ~ 89mm = 3.5in
    'figure.dpi': 150,              # Display DPI (save at 300+)
    'font.size': 10,
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'axes.titlesize': 12,
    'axes.labelsize': 11,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 9,
    'axes.linewidth': 1.0,
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'lines.linewidth': 1.5,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
}

# Apply temporarily using context manager
print('Publication parameters defined. Use with:\n'
      '  with plt.rc_context(pub_params):\n'
      '      fig, ax = plt.subplots()\n'
      '      ...')

### 13.2 Multi-Panel Publication Figure

In [ ]:
with plt.rc_context(pub_params):
    fig = plt.figure(figsize=(14, 10))
    gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.35)

    # Panel A: Volcano plot
    ax_a = fig.add_subplot(gs[0, 0])
    for cat in ['Not Significant', 'Up', 'Down']:
        sub = volcano_df[volcano_df['category'] == cat]
        ax_a.scatter(sub['log2FC'], sub['neg_log10_p'],
                    c=color_map[cat], s=5, alpha=0.5, edgecolors='none')
    ax_a.axhline(-np.log10(0.05), color='black', linestyle='--', linewidth=0.6)
    ax_a.axvline(1, color='black', linestyle='--', linewidth=0.6)
    ax_a.axvline(-1, color='black', linestyle='--', linewidth=0.6)
    ax_a.set_xlabel('log$_2$(Fold Change)')
    ax_a.set_ylabel('-log$_{10}$(p-value)')
    ax_a.set_title('Differential Expression')
    ax_a.text(-0.15, 1.05, 'A', transform=ax_a.transAxes,
             fontsize=16, fontweight='bold', va='bottom')

    # Panel B: PCA
    ax_b = fig.add_subplot(gs[0, 1])
    for cond in palette:
        sub = pca_df[pca_df['Condition'] == cond]
        ax_b.scatter(sub['PC1'], sub['PC2'], c=palette[cond],
                    marker=markers[cond], s=60, label=cond,
                    edgecolors='black', linewidth=0.5)
    ax_b.set_xlabel(f"PC1 ({pca.explained_variance_ratio_[0]*100:.0f}%)")
    ax_b.set_ylabel(f"PC2 ({pca.explained_variance_ratio_[1]*100:.0f}%)")
    ax_b.set_title('PCA of Samples')
    ax_b.legend(fontsize=7, frameon=True)
    ax_b.text(-0.15, 1.05, 'B', transform=ax_b.transAxes,
             fontsize=16, fontweight='bold', va='bottom')

    # Panel C: Expression barplot (top DEGs)
    ax_c = fig.add_subplot(gs[0, 2])
    top_genes = volcano_df[volcano_df['category'] != 'Not Significant'].nlargest(10, 'neg_log10_p')
    colors_bar = [color_map[c] for c in top_genes['category']]
    ax_c.barh(range(len(top_genes)), top_genes['log2FC'], color=colors_bar, edgecolor='black', linewidth=0.5)
    ax_c.set_yticks(range(len(top_genes)))
    ax_c.set_yticklabels(top_genes['gene'], fontsize=8)
    ax_c.set_xlabel('log$_2$(Fold Change)')
    ax_c.set_title('Top Differentially Expressed')
    ax_c.axvline(0, color='black', linewidth=0.8)
    ax_c.text(-0.15, 1.05, 'C', transform=ax_c.transAxes,
             fontsize=16, fontweight='bold', va='bottom')

    # Panel D: GC content
    ax_d = fig.add_subplot(gs[1, 0])
    ax_d.fill_between(positions[:len(gc_values)], gc_values, 50,
                      where=(gc_values >= 50), color='#E53935', alpha=0.3)
    ax_d.fill_between(positions[:len(gc_values)], gc_values, 50,
                      where=(gc_values < 50), color='#1E88E5', alpha=0.3)
    ax_d.plot(positions[:len(gc_values)], gc_values, 'k-', linewidth=0.5)
    ax_d.axhline(50, color='gray', linestyle='--', linewidth=0.6)
    ax_d.set_xlabel('Position (bp)')
    ax_d.set_ylabel('GC Content (%)')
    ax_d.set_title('GC Content Landscape')
    ax_d.text(-0.15, 1.05, 'D', transform=ax_d.transAxes,
             fontsize=16, fontweight='bold', va='bottom')

    # Panel E: Quality scores (simplified)
    ax_e = fig.add_subplot(gs[1, 1])
    medians = [np.median(quality_array[i, :]) for i in range(read_length)]
    q25 = [np.percentile(quality_array[i, :], 25) for i in range(read_length)]
    q75 = [np.percentile(quality_array[i, :], 75) for i in range(read_length)]
    ax_e.fill_between(base_positions, q25, q75, alpha=0.3, color='steelblue')
    ax_e.plot(base_positions, medians, color='steelblue', linewidth=1)
    ax_e.axhline(30, color='green', linestyle='--', linewidth=0.6)
    ax_e.axhline(20, color='orange', linestyle='--', linewidth=0.6)
    ax_e.set_xlabel('Position in Read')
    ax_e.set_ylabel('Phred Score')
    ax_e.set_title('Per-Base Quality')
    ax_e.set_ylim(0, 42)
    ax_e.text(-0.15, 1.05, 'E', transform=ax_e.transAxes,
             fontsize=16, fontweight='bold', va='bottom')

    # Panel F: Dendrogram
    ax_f = fig.add_subplot(gs[1, 2])
    dendrogram(Z, labels=species, ax=ax_f, leaf_rotation=90,
               leaf_font_size=7, color_threshold=25,
               above_threshold_color='gray')
    ax_f.set_ylabel('Distance')
    ax_f.set_title('Phylogenetic Tree')
    ax_f.text(-0.15, 1.05, 'F', transform=ax_f.transAxes,
             fontsize=16, fontweight='bold', va='bottom')

    plt.show()

    # How to save:
    # fig.savefig('Figure1.png', dpi=300, bbox_inches='tight')
    # fig.savefig('Figure1.svg', bbox_inches='tight')   # vector (editable)
    # fig.savefig('Figure1.pdf', bbox_inches='tight')   # vector (for LaTeX)

print('\nTo save this figure in publication formats:\n'
      "  fig.savefig('Figure1.png', dpi=300)  # raster, 300 DPI\n"
      "  fig.savefig('Figure1.svg')            # vector, editable in Illustrator\n"
      "  fig.savefig('Figure1.pdf')            # vector, for LaTeX")

### 13.3 Tips for Publication Figures

| Aspect | Recommendation |
|--------|---------------|
| **DPI** | 300 for raster (PNG), vector formats (SVG/PDF) preferred |
| **Font** | Arial or Helvetica (most journals accept these) |
| **Font size** | Minimum 6-8 pt after scaling to final size |
| **Colors** | Use colorblind-friendly palettes (viridis, cividis, Set2) |
| **Panel labels** | Bold uppercase A, B, C in the top-left corner |
| **Line width** | Minimum 0.5 pt for print visibility |
| **Figure width** | Single column: 89 mm (3.5"); double: 183 mm (7.2") |
| **File format** | SVG/PDF for line art; PNG/TIFF for images with gradients |

---
## 14. Exercises

Practice creating biological visualizations on your own.

### Exercise 1: Amino Acid Composition Bar Chart

Create a **grouped bar chart** comparing amino acid frequencies between two proteins.

Use these protein sequences:
```python
protein_1 = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
protein_2 = "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHL"
```

Requirements:
1. Count the frequency of each amino acid in both proteins
2. Plot only amino acids that appear at least once in either protein
3. Use grouped bars (side by side), with different colors for each protein
4. Add proper labels, title, and legend

In [ ]:
# YOUR CODE HERE


In [ ]:
# --- SOLUTION ---
from collections import Counter

protein_1 = "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSH"
protein_2 = "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHL"

counts_1 = Counter(protein_1)
counts_2 = Counter(protein_2)
all_aa = sorted(set(counts_1.keys()) | set(counts_2.keys()))

freq_1 = [counts_1.get(aa, 0) / len(protein_1) * 100 for aa in all_aa]
freq_2 = [counts_2.get(aa, 0) / len(protein_2) * 100 for aa in all_aa]

fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(all_aa))
width = 0.35

ax.bar(x - width/2, freq_1, width, label='Hemoglobin alpha', color='#2196F3', edgecolor='white')
ax.bar(x + width/2, freq_2, width, label='Myoglobin', color='#FF9800', edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(all_aa, fontsize=11, fontweight='bold')
ax.set_xlabel('Amino Acid', fontsize=12)
ax.set_ylabel('Frequency (%)', fontsize=12)
ax.set_title('Amino Acid Composition Comparison', fontsize=14)
ax.legend(fontsize=10)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

### Exercise 2: Codon Usage Heatmap

Create a **heatmap** of codon usage (RSCU -- Relative Synonymous Codon Usage) for two organisms.

Given data:
```python
# RSCU values for selected codons (simplified)
codons = ['UUU', 'UUC', 'UUA', 'UUG', 'CUU', 'CUC', 'CUA', 'CUG',
          'AUU', 'AUC', 'AUA', 'GUU', 'GUC', 'GUA', 'GUG']
ecoli_rscu = [1.8, 0.2, 0.5, 0.5, 0.4, 0.4, 0.1, 2.1,
              1.5, 1.2, 0.3, 1.6, 0.2, 1.0, 1.2]
human_rscu = [0.9, 1.1, 0.4, 0.8, 0.8, 1.2, 0.4, 2.4,
              1.1, 1.4, 0.5, 0.7, 0.9, 0.5, 1.9]
```

Requirements:
1. Create a DataFrame with codons as rows, organisms as columns
2. Use `sns.heatmap()` with annotations (show values)
3. Use a diverging colormap centered at 1.0 (RSCU=1 means no bias)
4. Add proper title and labels

In [ ]:
# YOUR CODE HERE


In [ ]:
# --- SOLUTION ---
codons = ['UUU', 'UUC', 'UUA', 'UUG', 'CUU', 'CUC', 'CUA', 'CUG',
          'AUU', 'AUC', 'AUA', 'GUU', 'GUC', 'GUA', 'GUG']
ecoli_rscu = [1.8, 0.2, 0.5, 0.5, 0.4, 0.4, 0.1, 2.1,
              1.5, 1.2, 0.3, 1.6, 0.2, 1.0, 1.2]
human_rscu = [0.9, 1.1, 0.4, 0.8, 0.8, 1.2, 0.4, 2.4,
              1.1, 1.4, 0.5, 0.7, 0.9, 0.5, 1.9]

rscu_df = pd.DataFrame({'E. coli': ecoli_rscu, 'Human': human_rscu}, index=codons)

fig, ax = plt.subplots(figsize=(5, 10))
sns.heatmap(rscu_df, annot=True, fmt='.1f', cmap='RdBu_r', center=1.0,
            vmin=0, vmax=2.5, linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'RSCU', 'shrink': 0.6})
ax.set_title('Relative Synonymous Codon Usage (RSCU)', fontsize=13, pad=10)
ax.set_ylabel('Codon', fontsize=12)
ax.set_xlabel('Organism', fontsize=12)

plt.tight_layout()
plt.show()

### Exercise 3: MA Plot from Simulated RNA-seq

An **MA plot** shows log2 fold-change (M) vs. average expression (A). It is an alternative to the volcano plot.

Requirements:
1. Generate simulated data for 3000 genes: random `base_mean` (log-normal) and `log2FC` (normal, mostly near 0)
2. Inject 50 upregulated and 50 downregulated genes
3. Color significant genes differently
4. Use a **loess/lowess smoothing line** (hint: `from statsmodels.nonparametric.smoothers_lowess import lowess` or just `np.polyfit`)
5. Add horizontal line at y=0

In [ ]:
# YOUR CODE HERE


In [ ]:
# --- SOLUTION ---
np.random.seed(123)
n = 3000

base_mean = np.random.lognormal(5, 2, n)
log2fc = np.random.normal(0, 0.3, n)

# Inject DE genes
up_idx = np.random.choice(n, 50, replace=False)
log2fc[up_idx] = np.abs(np.random.normal(2, 0.5, 50))
remaining = list(set(range(n)) - set(up_idx))
down_idx = np.random.choice(remaining, 50, replace=False)
log2fc[down_idx] = -np.abs(np.random.normal(2, 0.5, 50))

significant = np.zeros(n, dtype=bool)
significant[up_idx] = True
significant[down_idx] = True

A = np.log10(base_mean)
M = log2fc

fig, ax = plt.subplots(figsize=(10, 6))

ax.scatter(A[~significant], M[~significant], s=8, alpha=0.3, c='gray', label='Not significant')
ax.scatter(A[significant & (M > 0)], M[significant & (M > 0)],
           s=20, alpha=0.7, c='red', label='Upregulated')
ax.scatter(A[significant & (M < 0)], M[significant & (M < 0)],
           s=20, alpha=0.7, c='blue', label='Downregulated')

# Trend line
sort_idx = np.argsort(A)
z = np.polyfit(A[sort_idx], M[sort_idx], 3)
p = np.poly1d(z)
x_smooth = np.linspace(A.min(), A.max(), 200)
ax.plot(x_smooth, p(x_smooth), 'k-', linewidth=1.5, alpha=0.6)

ax.axhline(0, color='black', linestyle='--', linewidth=0.8)
ax.set_xlabel('A: log$_{10}$(Base Mean Expression)', fontsize=12)
ax.set_ylabel('M: log$_2$(Fold Change)', fontsize=12)
ax.set_title('MA Plot', fontsize=14)
ax.legend(fontsize=10)

plt.tight_layout()
plt.show()

### Exercise 4: Multi-Panel Restriction Enzyme Analysis

Create a **2-panel figure** analyzing a DNA sequence:

```python
sequence = "ATGCGAATTCGATCGATCGATATCGCGATCGATCGGAATTCATCGATCGATCGATCGATATCGAATTCGATCGAT" * 20
```

**Panel A**: Bar chart of restriction enzyme cut counts
- Count occurrences of: `GAATTC` (EcoRI), `GATATC` (EcoRV), `AGATCT` (BglII), `GGATCC` (BamHI)

**Panel B**: Position of EcoRI sites along the sequence
- Use vertical lines (`ax.axvline`) or a stem plot to mark each site

In [ ]:
# YOUR CODE HERE


In [ ]:
# --- SOLUTION ---
sequence = "ATGCGAATTCGATCGATCGATATCGCGATCGATCGGAATTCATCGATCGATCGATCGATATCGAATTCGATCGAT" * 20

enzymes = {
    'EcoRI (GAATTC)': 'GAATTC',
    'EcoRV (GATATC)': 'GATATC',
    'BglII (AGATCT)': 'AGATCT',
    'BamHI (GGATCC)': 'GGATCC',
}

# Count occurrences
counts = {}
for name, site in enzymes.items():
    counts[name] = sequence.count(site)

# Find EcoRI positions
ecori_positions = []
start = 0
while True:
    idx = sequence.find('GAATTC', start)
    if idx == -1:
        break
    ecori_positions.append(idx)
    start = idx + 1

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 7), height_ratios=[1, 1])

# Panel A: Bar chart
colors = ['#E53935', '#2196F3', '#4CAF50', '#FF9800']
bars = ax1.bar(list(counts.keys()), list(counts.values()), color=colors, edgecolor='black', linewidth=0.8)
ax1.set_ylabel('Number of Cut Sites', fontsize=12)
ax1.set_title('Restriction Enzyme Cut Site Frequency', fontsize=13)
for bar, count in zip(bars, counts.values()):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
             str(count), ha='center', fontsize=11, fontweight='bold')
ax1.text(-0.05, 1.05, 'A', transform=ax1.transAxes, fontsize=15, fontweight='bold')

# Panel B: EcoRI site positions
markerline, stemlines, baseline = ax2.stem(ecori_positions, [1]*len(ecori_positions))
plt.setp(stemlines, color='#E53935', linewidth=1.5)
plt.setp(markerline, color='#E53935', markersize=5)
plt.setp(baseline, color='black', linewidth=0.5)
ax2.set_xlabel('Position in Sequence (bp)', fontsize=12)
ax2.set_ylabel('')
ax2.set_yticks([])
ax2.set_title(f'EcoRI (GAATTC) Cut Site Positions ({len(ecori_positions)} sites)', fontsize=13)
ax2.set_xlim(0, len(sequence))
ax2.text(-0.05, 1.05, 'B', transform=ax2.transAxes, fontsize=15, fontweight='bold')

plt.tight_layout()
plt.show()

### Exercise 5: Survival Curve (Kaplan-Meier Style)

Create a **Kaplan-Meier survival curve** comparing two treatment groups.

Given data (time in months, whether the event occurred):
```python
# Treatment A: slower progression
time_a = [1, 3, 5, 7, 9, 12, 15, 18, 22, 25, 30, 36, 40, 48]
event_a = [0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]  # 1 = event (death/relapse)

# Treatment B: faster progression
time_b = [1, 2, 4, 5, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24]
event_b = [0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0]
```

Requirements:
1. Calculate survival probability at each time point: S(t) = (n_at_risk - n_events) / n_at_risk
2. Plot as **step functions** (`ax.step()` with `where='post'`)
3. Mark censored observations with small ticks
4. Add legend, labels, and title

In [ ]:
# YOUR CODE HERE


In [ ]:
# --- SOLUTION ---
def kaplan_meier(times, events):
    """Simple Kaplan-Meier estimator."""
    n = len(times)
    km_times = [0]
    km_survival = [1.0]
    censored_times = []
    censored_surv = []
    
    at_risk = n
    survival = 1.0
    
    for t, e in zip(times, events):
        if e == 1:
            survival *= (at_risk - 1) / at_risk
            km_times.append(t)
            km_survival.append(survival)
        else:
            censored_times.append(t)
            censored_surv.append(survival)
        at_risk -= 1
    
    return km_times, km_survival, censored_times, censored_surv

time_a = [1, 3, 5, 7, 9, 12, 15, 18, 22, 25, 30, 36, 40, 48]
event_a = [0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]
time_b = [1, 2, 4, 5, 6, 8, 10, 12, 14, 16, 18, 20, 22, 24]
event_b = [0, 1, 1, 1, 0, 1, 1, 0, 1, 1, 0, 1, 1, 0]

km_t_a, km_s_a, cens_t_a, cens_s_a = kaplan_meier(time_a, event_a)
km_t_b, km_s_b, cens_t_b, cens_s_b = kaplan_meier(time_b, event_b)

fig, ax = plt.subplots(figsize=(10, 6))

ax.step(km_t_a, km_s_a, where='post', color='#2196F3', linewidth=2, label='Treatment A')
ax.step(km_t_b, km_s_b, where='post', color='#E53935', linewidth=2, label='Treatment B')

# Censored markers
ax.plot(cens_t_a, cens_s_a, '|', color='#2196F3', markersize=10, markeredgewidth=2)
ax.plot(cens_t_b, cens_s_b, '|', color='#E53935', markersize=10, markeredgewidth=2)

ax.set_xlabel('Time (months)', fontsize=12)
ax.set_ylabel('Survival Probability', fontsize=12)
ax.set_title('Kaplan-Meier Survival Curves', fontsize=14)
ax.set_ylim(0, 1.05)
ax.set_xlim(0, 50)
ax.legend(fontsize=11)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

# Add "at risk" annotation
ax.text(0.6, 0.2, 'Tick marks = censored\nobservations',
        transform=ax.transAxes, fontsize=10,
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

plt.tight_layout()
plt.show()

### Exercise 6: Create Your Own Publication Figure

**Challenge**: Combine **any 3 of the plot types** from this notebook into a single multi-panel publication figure.

Requirements:
1. Use `plt.figure()` with `add_gridspec()` or `plt.subplots()`
2. Apply `pub_params` via `plt.rc_context()`
3. Add panel labels (A, B, C)
4. Use a coherent color scheme
5. The figure should tell a biological story (e.g., "Characterization of Drug Response")

In [ ]:
# YOUR CODE HERE -- be creative!


---
## Summary

### Key Takeaways

| Library | When to Use | Key Functions |
|---------|-------------|---------------|
| **Matplotlib** | Full control, custom layouts | `plt.subplots()`, `ax.plot()`, `ax.scatter()`, `ax.bar()`, `fig.savefig()` |
| **Seaborn** | Statistical plots, quick exploration | `sns.heatmap()`, `sns.clustermap()`, `sns.violinplot()`, `sns.pairplot()` |
| **Plotly** | Interactive exploration | `px.scatter()`, `go.Figure()` |

### Bioinformatics Visualization Cheat Sheet

| Data Type | Best Plot |
|-----------|-----------|
| Differential expression | Volcano plot, MA plot |
| Expression across samples | Heatmap with clustering |
| Sample relationships | PCA scatter |
| Sequence features | GC landscape, coverage plot |
| Length distributions | Histogram, KDE |
| Group comparisons | Box plot, violin plot |
| Evolutionary relationships | Dendrogram |
| Sequencing quality | FastQC-style per-base quality |
| Survival analysis | Kaplan-Meier step plot |

### Publication Checklist

- [ ] DPI >= 300 for raster; prefer SVG/PDF for vector
- [ ] Font: Arial or Helvetica, minimum 6-8 pt after scaling
- [ ] Colorblind-friendly palette (viridis, cividis, colorblind)
- [ ] Panel labels (A, B, C) in bold
- [ ] All axes labeled with units
- [ ] Legend placed without overlapping data
- [ ] `bbox_inches='tight'` when saving

---

[← Previous: Module 17: Data Wrangling with Pandas](../../Tier_1_Python_for_Bioinformatics/17_Data_Wrangling/01_data_wrangling.ipynb) | [Next: Tier 2: Core Bioinformatics -- Skills Check →](../../Tier_2_Core_Bioinformatics/00_Skills_Check/00_skills_check.ipynb)